In [0]:
%sql
use catalog test;
use schema sql;

In [0]:
%sql
-- Set-up the tables
DROP TABLE IF EXISTS sales;
CREATE TABLE sales
(
    company    VARCHAR(20),
    year       INTEGER,
    quarter    CHAR(2),
    volume     INTEGER
);
INSERT INTO sales (company, year, quarter, volume)
VALUES
    ('BYD', 2022,'Q1',143223),
    ('BYD', 2022,'Q2',180296),
    ('BYD', 2022,'Q3',258610),
    ('BYD', 2022,'Q4',329011),
    ('BYD', 2023,'Q1',264647),
    ('BYD', 2023,'Q2',352163),
    ('BYD', 2023,'Q3',431603),
    ('BYD', 2023,'Q4',526409),    
    ('Tesla', 2022,'Q1',310048),
    ('Tesla', 2022,'Q2',254695),
    ('Tesla', 2022,'Q3',343830),
    ('Tesla', 2022,'Q4',405278),
    ('Tesla', 2023,'Q1',422875),
    ('Tesla', 2023,'Q2',466140),
    ('Tesla', 2023,'Q3',435059),
    ('Tesla', 2023,'Q4',484507);	

SELECT * FROM sales;

company,year,quarter,volume
BYD,2022,Q1,143223
BYD,2022,Q2,180296
BYD,2022,Q3,258610
BYD,2022,Q4,329011
BYD,2023,Q1,264647
BYD,2023,Q2,352163
BYD,2023,Q3,431603
BYD,2023,Q4,526409
Tesla,2022,Q1,310048
Tesla,2022,Q2,254695


In [0]:
# Create DataFrame using schema inference (Spark Session)
# Spark automatically determines the data types based on the input data instead of manually injecting the schema.

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
data = [
    ("BYD", 2022, "Q1", 143223),
    ("BYD", 2022, "Q2", 180296),
    ("BYD", 2022, "Q3", 258610),
    ("BYD", 2022, "Q4", 329011),
    ("BYD", 2023, "Q1", 264647),
    ("BYD", 2023, "Q2", 352163),
    ("BYD", 2023, "Q3", 431603),
    ("BYD", 2023, "Q4", 526409),
    ("Tesla", 2022, "Q1", 310048),
    ("Tesla", 2022, "Q2", 254695),
    ("Tesla", 2022, "Q3", 343830),
    ("Tesla", 2022, "Q4", 405278),
    ("Tesla", 2023, "Q1", 422875),
    ("Tesla", 2023, "Q2", 466140),
    ("Tesla", 2023, "Q3", 435059),
    ("Tesla", 2023, "Q4", 484507)
]
columns = ["company", "year", "quarter", "volume"]
sales = spark.createDataFrame(data, columns)
sales.display()
sales.printSchema()

company,year,quarter,volume
BYD,2022,Q1,143223
BYD,2022,Q2,180296
BYD,2022,Q3,258610
BYD,2022,Q4,329011
BYD,2023,Q1,264647
BYD,2023,Q2,352163
BYD,2023,Q3,431603
BYD,2023,Q4,526409
Tesla,2022,Q1,310048
Tesla,2022,Q2,254695


root
 |-- company: string (nullable = true)
 |-- year: long (nullable = true)
 |-- quarter: string (nullable = true)
 |-- volume: long (nullable = true)



In [0]:
# Create DataFrame using an explicit schema. 
# Where we define the data types & make the schema 
# Recommended for production ETL pipelines and large datasets.

from pyspark.sql.types import StructType, StructField, StringType, IntegerType
schema = StructType([
    StructField("company", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("quarter", StringType(), True),
    StructField("volume", IntegerType(), True)
])
sales_df = spark.createDataFrame(data, schema)
display(sales_df)
sales_df.printSchema()
sales_df.write.mode("overwrite").saveAsTable("test.pyspark.sales")

company,year,quarter,volume
BYD,2022,Q1,143223
BYD,2022,Q2,180296
BYD,2022,Q3,258610
BYD,2022,Q4,329011
BYD,2023,Q1,264647
BYD,2023,Q2,352163
BYD,2023,Q3,431603
BYD,2023,Q4,526409
Tesla,2022,Q1,310048
Tesla,2022,Q2,254695


root
 |-- company: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: string (nullable = true)
 |-- volume: integer (nullable = true)



In [0]:
from pyspark.sql.functions import col, sum
from pyspark.sql.window import Window

sales_df.withColumn(
    "sales_per_quarter",
    sum(col("volume")).over(Window.partitionBy(col("company"), col("year")))
).display()

company,year,quarter,volume,sales_per_quarter
BYD,2022,Q1,143223,911140
BYD,2022,Q2,180296,911140
BYD,2022,Q3,258610,911140
BYD,2022,Q4,329011,911140
BYD,2023,Q1,264647,1574822
BYD,2023,Q2,352163,1574822
BYD,2023,Q3,431603,1574822
BYD,2023,Q4,526409,1574822
Tesla,2022,Q1,310048,1313851
Tesla,2022,Q2,254695,1313851


In [0]:
%sql
select 
    *,
    sum(volume) over (partition by company, year)
    as total_volume
from sales

company,year,quarter,volume,total_volume
BYD,2022,Q1,143223,911140
BYD,2022,Q2,180296,911140
BYD,2022,Q3,258610,911140
BYD,2022,Q4,329011,911140
BYD,2023,Q1,264647,1574822
BYD,2023,Q2,352163,1574822
BYD,2023,Q3,431603,1574822
BYD,2023,Q4,526409,1574822
Tesla,2022,Q1,310048,1313851
Tesla,2022,Q2,254695,1313851


##### LAG function

###### LAG(column, offset, default) --> returns the value from a previous row based on the ORDER BY sequence. 
- 'offset' (default = 1) --> specifies how many rows back to look.
- 'default' --> We can mention it manually to replace NULL when no previous row exists. 

###### It must be used with OVER(ORDER BY ...), and PARTITION BY (optional) resets the calculation for each group. Commonly used for MoM, YoY, QoQ growth and comparing current values with previous records.

In [0]:
%sql
select
    *,
    lag(volume, 4, 0) over(order by year, quarter) as prev_volume,
    volume - lag(volume,4,0) over(order by year, quarter) as growth
from sales
where company = "Tesla";

company,year,quarter,volume,prev_volume,growth
Tesla,2022,Q1,310048,0,310048
Tesla,2022,Q2,254695,0,254695
Tesla,2022,Q3,343830,0,343830
Tesla,2022,Q4,405278,0,405278
Tesla,2023,Q1,422875,310048,112827
Tesla,2023,Q2,466140,254695,211445
Tesla,2023,Q3,435059,343830,91229
Tesla,2023,Q4,484507,405278,79229


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

sales_df.withColumn(
    "prev_quarter_sales",
    lag(col("volume"),1,0).over(Window.orderBy(col("year"), col("quarter")))
).withColumn(
    "growth",
    (col("volume") - col("prev_quarter_sales"))
).filter(col("company") == "Tesla").display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


company,year,quarter,volume,prev_quarter_sales,growth
Tesla,2022,Q1,310048,143223,166825
Tesla,2022,Q2,254695,180296,74399
Tesla,2022,Q3,343830,258610,85220
Tesla,2022,Q4,405278,329011,76267
Tesla,2023,Q1,422875,264647,158228
Tesla,2023,Q2,466140,352163,113977
Tesla,2023,Q3,435059,431603,3456
Tesla,2023,Q4,484507,526409,-41902


##### LEAD function

###### LEAD(column, offset, default) --> returns the value from a next row based on the ORDER BY sequence.
- 'offset' (default = 1) --> specifies how many rows ahead to look.
- 'default' --> We can mention it manually to replace NULL when no next row exists.

###### It must be used with OVER(ORDER BY ...), and PARTITION BY (optional) resets the calculation for each group. Commonly used to compare the current row with future records, calculate forward differences, and identify the next event or transaction.

In [0]:
%sql
select
    *,
    lead(volume,1,0) over(order by year, quarter) as next_volume
from sales
where company = "Tesla";

company,year,quarter,volume,next_volume
Tesla,2022,Q1,310048,254695
Tesla,2022,Q2,254695,343830
Tesla,2022,Q3,343830,405278
Tesla,2022,Q4,405278,422875
Tesla,2023,Q1,422875,466140
Tesla,2023,Q2,466140,435059
Tesla,2023,Q3,435059,484507
Tesla,2023,Q4,484507,0


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

sales_df.withColumn(
    "next_volume",
    lead(col("volume"),1,0).over(Window.orderBy(col("year"), col("quarter")))
).withColumn(
    "growth",
    (col("next_volume") - col("volume"))
).filter(
    col("company") == "Tesla"
).display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


company,year,quarter,volume,next_volume,growth
Tesla,2022,Q1,310048,180296,-129752
Tesla,2022,Q2,254695,258610,3915
Tesla,2022,Q3,343830,329011,-14819
Tesla,2022,Q4,405278,264647,-140631
Tesla,2023,Q1,422875,352163,-70712
Tesla,2023,Q2,466140,431603,-34537
Tesla,2023,Q3,435059,526409,91350
Tesla,2023,Q4,484507,0,-484507
